In [24]:
import sys
import os
from os.path import join
import glob
from copy import deepcopy

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display, clear_output
from tqdm import tqdm

sys.path.insert(0, "../../ABC-SN/code")
import abcsn_config
import abcsn_training
import data_degrading as dg
import data_plotting as dplt
import data_preparation as dp
import preprocessing

sys.path.insert(0, "../code")
import review_spectrum as rs
import spectral_features as sf
import measure_signal as ms

from icecream import ic
from importlib import reload

In [37]:
REPO_DIR = "../"
DATA_DIR = join(REPO_DIR, "data")

In [63]:
file_SNRmetadata = join(DATA_DIR, "forSNR", "data_metadata_SNRinfo.parquet")
df_dataset = pd.read_parquet(file_SNRmetadata)

In [64]:
df_dataset["Denoising Parameter"].value_counts(dropna=False)

Denoising Parameter
 10.0     2327
 20.0      744
 15.0      341
-999.0     188
 30.0      159
 25.0        2
 50.0        1
Name: count, dtype: int64

In [65]:
inds_to_drop = df_dataset.index[df_dataset["Denoising Parameter"] == -999].to_numpy()
inds_to_drop

array([   4,    5,    8,   10,   11,   12,   13,   14,   15,   16,   17,
         18,   20,   21,   22,   23,   24,   25,   26,   27,   28,   29,
         30,   31,   32,   33,   46,   48,   49,   50,   51,   52,   53,
         61,   62,   63,   64,   65,   66,   67,   68,   69,   70,   73,
         74,  124,  133,  134,  143,  145,  215,  218,  220,  221,  223,
        225,  243,  244,  254,  314,  321,  365,  366,  384,  389,  392,
        399,  428,  429,  448,  450,  451,  452,  453,  498,  513,  662,
        666,  711,  759,  800,  801,  846,  847,  849,  856,  883,  892,
        893,  979,  980, 1011, 1022, 1109, 1224, 1230, 1667, 1668, 1831,
       1842, 1891, 1894, 1897, 1898, 2343, 2371, 2382, 2466, 2467, 2468,
       2469, 2526, 2532, 2534, 2535, 2536, 2537, 2551, 2552, 2718, 2729,
       2730, 2731, 2732, 2733, 2734, 2794, 2795, 2796, 2797, 2798, 2799,
       2800, 2801, 3030, 3321, 3437, 3461, 3463, 3490, 3491, 3496, 3497,
       3498, 3500, 3504, 3505, 3507, 3508, 3509, 35

In [66]:
df_dataset_clean = df_dataset.drop(index=inds_to_drop)
df_dataset_clean

,name_prefix,name_year,name_suffix,SN Name,SN Subtype,SN Subtype ID,SN Maintype,SN Maintype ID,Spectral Phase,Spectrum Cardinality,...,9872.21,9885.59,9898.98,9912.39,9925.82,9939.27,9952.73,9966.21,9979.71,9993.24
0,sn,1981,B,sn1981B,Ia-norm,0,Ia,0,-1.50,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,sn,1981,B,sn1981B,Ia-norm,0,Ia,0,15.40,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,sn,1981,B,sn1981B,Ia-norm,0,Ia,0,18.30,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,sn,1981,B,sn1981B,Ia-norm,0,Ia,0,22.30,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,sn,1983,N,sn1983N,Ib-norm,4,Ib,1,4.00,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3756,unknown,2017,ein,17ein,Ic-norm,7,Ic,2,12.30,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3758,unknown,2017,ein,17ein,Ic-norm,7,Ic,2,18.29,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3759,unknown,2017,ein,17ein,Ic-norm,7,Ic,2,22.27,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3760,unknown,2017,ein,17ein,Ic-norm,7,Ic,2,38.24,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [67]:
df_dataset_clean = df_dataset_clean.sort_values(["SN Subtype ID", "name_year", "name_suffix", "Spectral Phase", "Spectrum Cardinality"]).reset_index(drop=True)
df_dataset_clean

,name_prefix,name_year,name_suffix,SN Name,SN Subtype,SN Subtype ID,SN Maintype,SN Maintype ID,Spectral Phase,Spectrum Cardinality,...,9872.21,9885.59,9898.98,9912.39,9925.82,9939.27,9952.73,9966.21,9979.71,9993.24
0,sn,1981,B,sn1981B,Ia-norm,0,Ia,0,-1.5,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,sn,1981,B,sn1981B,Ia-norm,0,Ia,0,15.4,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,sn,1981,B,sn1981B,Ia-norm,0,Ia,0,18.3,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,sn,1981,B,sn1981B,Ia-norm,0,Ia,0,22.3,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,sn,1989,B,sn1989B,Ia-norm,0,Ia,0,-7.2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3569,sn,2006,bp,sn2006bp,IIP,9,II,3,17.2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3570,sn,2006,bp,sn2006bp,IIP,9,II,3,20.1,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3571,sn,2006,bp,sn2006bp,IIP,9,II,3,25.2,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3572,sn,2006,bp,sn2006bp,IIP,9,II,3,34.1,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [82]:
df_dataset_clean.to_parquet(join(DATA_DIR, "forSNR", "final_full_dataset.parquet"))